# Dataset Expansion Notebook

This notebook expands a wide dataset from 680 to 10,000 columns using various transformations:
- Positive and negative correlations
- Column combinations
- Noise injection
- Scaling transformations

All generated columns maintain realistic stock-like values.

In [ ]:
import pandas as pd
import numpy as np
import random
import string
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

In [ ]:
def generate_random_column_name(length=6):
    """Generate random stock-like column names"""
    prefixes = ['STOCK', 'ASSET', 'FUND', 'BOND', 'REIT', 'ETF', 'INDEX', 'COMP']
    suffixes = ['_RET', '_VOL', '_BETA', '_PRICE', '_YIELD', '_CAP', '_RATIO', '_SCORE']
    
    if random.random() < 0.6:
        # Stock ticker style
        ticker = ''.join(random.choices(string.ascii_uppercase, k=random.randint(3, 5)))
        if random.random() < 0.3:
            ticker += random.choice(suffixes)
        return ticker
    else:
        # Descriptive name style
        prefix = random.choice(prefixes)
        number = random.randint(1, 999)
        suffix = random.choice(suffixes) if random.random() < 0.5 else ''
        return f"{prefix}_{number}{suffix}"

In [ ]:
def normalize_to_stock_range(data, min_val=-0.15, max_val=0.15):
    """Normalize data to typical stock return range"""
    data_norm = (data - data.min()) / (data.max() - data.min())
    return data_norm * (max_val - min_val) + min_val

In [ ]:
def create_correlated_features(base_data, n_features, correlation_strength=0.7):
    """Create features with specified correlation to base data"""
    new_features = pd.DataFrame()
    
    for i in range(n_features):
        # Choose random base column
        base_col = np.random.choice(base_data.columns)
        base_values = base_data[base_col].values
        
        # Generate noise
        noise = np.random.normal(0, 1, len(base_values))
        
        # Create correlated feature
        if correlation_strength > 0:
            # Positive correlation
            new_values = correlation_strength * base_values + np.sqrt(1 - correlation_strength**2) * noise
        else:
            # Negative correlation
            new_values = abs(correlation_strength) * (-base_values) + np.sqrt(1 - correlation_strength**2) * noise
        
        # Normalize to stock-like range
        new_values = normalize_to_stock_range(new_values)
        
        col_name = generate_random_column_name()
        new_features[col_name] = new_values
    
    return new_features

In [ ]:
def create_combined_features(base_data, n_features):
    """Create features by combining existing columns"""
    new_features = pd.DataFrame()
    
    for i in range(n_features):
        # Choose 2-4 random columns to combine
        n_cols = np.random.randint(2, 5)
        selected_cols = np.random.choice(base_data.columns, n_cols, replace=False)
        
        # Choose combination method
        method = np.random.choice(['weighted_sum', 'product', 'ratio', 'difference'])
        
        if method == 'weighted_sum':
            weights = np.random.uniform(-1, 1, n_cols)
            new_values = np.sum([w * base_data[col].values for w, col in zip(weights, selected_cols)], axis=0)
        
        elif method == 'product':
            new_values = np.prod([base_data[col].values for col in selected_cols], axis=0)
            new_values = np.sign(new_values) * np.log1p(np.abs(new_values))
        
        elif method == 'ratio':
            if n_cols >= 2:
                num = base_data[selected_cols[0]].values
                denom = base_data[selected_cols[1]].values + 0.001  # Avoid division by zero
                new_values = num / denom
            else:
                new_values = base_data[selected_cols[0]].values
        
        elif method == 'difference':
            if n_cols >= 2:
                new_values = base_data[selected_cols[0]].values - base_data[selected_cols[1]].values
            else:
                new_values = base_data[selected_cols[0]].values
        
        # Normalize to stock-like range
        new_values = normalize_to_stock_range(new_values)
        
        col_name = generate_random_column_name()
        new_features[col_name] = new_values
    
    return new_features

In [ ]:
def create_pca_features(base_data, n_components, n_features):
    """Create features using PCA components"""
    pca = PCA(n_components=n_components)
    pca_data = pca.fit_transform(base_data)
    
    new_features = pd.DataFrame()
    
    for i in range(n_features):
        # Select random PCA components and weights
        n_comp_selected = np.random.randint(1, min(n_components, 5) + 1)
        selected_components = np.random.choice(n_components, n_comp_selected, replace=False)
        weights = np.random.uniform(-1, 1, n_comp_selected)
        
        # Create new feature as weighted sum of PCA components
        new_values = np.sum([w * pca_data[:, comp] for w, comp in zip(weights, selected_components)], axis=0)
        
        # Add some noise
        noise = np.random.normal(0, 0.1, len(new_values))
        new_values = new_values + noise
        
        # Normalize to stock-like range
        new_values = normalize_to_stock_range(new_values)
        
        col_name = generate_random_column_name()
        new_features[col_name] = new_values
    
    return new_features

In [ ]:
def create_noise_features(base_data, n_features):
    """Create features with controlled noise patterns"""
    new_features = pd.DataFrame()
    
    for i in range(n_features):
        # Different noise patterns
        noise_type = np.random.choice(['gaussian', 'uniform', 'exponential', 'laplace'])
        n_samples = len(base_data)
        
        if noise_type == 'gaussian':
            new_values = np.random.normal(0, 0.05, n_samples)
        elif noise_type == 'uniform':
            new_values = np.random.uniform(-0.1, 0.1, n_samples)
        elif noise_type == 'exponential':
            new_values = np.random.exponential(0.02, n_samples)
            new_values = new_values * np.random.choice([-1, 1], n_samples)
        elif noise_type == 'laplace':
            new_values = np.random.laplace(0, 0.03, n_samples)
        
        # Normalize to stock-like range
        new_values = normalize_to_stock_range(new_values)
        
        col_name = generate_random_column_name()
        new_features[col_name] = new_values
    
    return new_features

In [ ]:
def expand_dataset(data, target_columns=10000):
    """Main function to expand dataset from 680 to target number of columns"""
    print(f"Starting with {data.shape[1]} columns")
    print(f"Target: {target_columns} columns")
    
    expanded_data = data.copy()
    columns_to_add = target_columns - data.shape[1]
    
    # Distribution of new features
    n_positive_corr = int(columns_to_add * 0.25)  # 25% positive correlations
    n_negative_corr = int(columns_to_add * 0.25)  # 25% negative correlations
    n_combined = int(columns_to_add * 0.30)       # 30% combined features
    n_pca = int(columns_to_add * 0.10)            # 10% PCA features
    n_noise = columns_to_add - (n_positive_corr + n_negative_corr + n_combined + n_pca)  # Remaining as noise
    
    print(f"Creating {n_positive_corr} positive correlation features...")
    pos_corr_features = create_correlated_features(data, n_positive_corr, 0.7)
    expanded_data = pd.concat([expanded_data, pos_corr_features], axis=1)
    
    print(f"Creating {n_negative_corr} negative correlation features...")
    neg_corr_features = create_correlated_features(data, n_negative_corr, -0.6)
    expanded_data = pd.concat([expanded_data, neg_corr_features], axis=1)
    
    print(f"Creating {n_combined} combined features...")
    combined_features = create_combined_features(data, n_combined)
    expanded_data = pd.concat([expanded_data, combined_features], axis=1)
    
    print(f"Creating {n_pca} PCA-based features...")
    pca_features = create_pca_features(data, min(50, data.shape[1]//2), n_pca)
    expanded_data = pd.concat([expanded_data, pca_features], axis=1)
    
    print(f"Creating {n_noise} noise features...")
    noise_features = create_noise_features(data, n_noise)
    expanded_data = pd.concat([expanded_data, noise_features], axis=1)
    
    print(f"Final dataset shape: {expanded_data.shape}")
    return expanded_data

## Example Usage

Load your wide dataset and expand it:

In [ ]:
# Load your data (replace with your actual data loading)
# data = pd.read_csv('your_wide_dataset.csv')

# For demonstration, create a sample dataset with 680 columns
print("Creating sample dataset for demonstration...")
sample_data = pd.DataFrame(
    np.random.uniform(-0.1, 0.1, (1000, 680)),
    columns=[f'STOCK_{i:03d}' for i in range(680)]
)

print(f"Sample data shape: {sample_data.shape}")
print(f"Sample data range: {sample_data.min().min():.4f} to {sample_data.max().max():.4f}")

In [ ]:
# Expand the dataset
expanded_dataset = expand_dataset(sample_data, target_columns=10000)

print("\nDataset expansion completed!")
print(f"Original shape: {sample_data.shape}")
print(f"Expanded shape: {expanded_dataset.shape}")
print(f"Value range: {expanded_dataset.min().min():.4f} to {expanded_dataset.max().max():.4f}")

In [ ]:
# Display some statistics
print("Sample column names from expanded dataset:")
print(expanded_dataset.columns[-20:].tolist())  # Last 20 column names

print("\nBasic statistics:")
print(expanded_dataset.describe().iloc[:, -5:])  # Stats for last 5 columns

In [ ]:
# Save the expanded dataset to data folder as synthetic_close.pkl
import os
os.makedirs('../../data', exist_ok=True)
expanded_dataset.to_pickle('../../data/synthetic_close.pkl')
print("Expanded dataset saved as 'quantum_portfolio/data/synthetic_close.pkl'")

print("Notebook execution completed successfully!")
print(f"Successfully expanded dataset from {sample_data.shape[1]} to {expanded_dataset.shape[1]} columns")

## Notes

- All generated features maintain stock-like value ranges (typically -15% to +15%)
- Column names are randomly generated in stock ticker style
- Different correlation levels and combination methods ensure diversity
- The expansion maintains statistical properties suitable for financial analysis
- Random seeds are set for reproducibility

In [ ]:
# Compare performance across different dataset sizes
dataset_comparison_solvers = [
    {'path_template': '../../results/qubo/results_wide_{}_dwave_cqm.pkl', 'name': 'D-Wave CQM (680 assets)'},
    {'path_template': '../../results/qubo/results_synthetic_{}_dwave_cqm.pkl', 'name': 'D-Wave CQM (10,000 synthetic)'},
]

print("\n" + "="*120)
print("DATASET SIZE COMPARISON - D-Wave CQM QUBO Optimization Performance")
print("="*120)

summary_df_comparison = performance_summary(dataset_comparison_solvers, n_periods)
if not summary_df_comparison.empty:
    display(summary_df_comparison)
    
    # Detailed comparison
    print("\nDataset Size Impact Analysis:")
    print("-" * 80)
    
    if len(summary_df_comparison) >= 2:
        wide_row = summary_df_comparison.iloc[0]
        synthetic_row = summary_df_comparison.iloc[1]
        
        print(f"680 Assets Dataset:")
        print(f"  • Average Return: {wide_row['Avg_Return']:.4f}")
        print(f"  • Execution Time: {wide_row['Avg_Time']:.2f}s")
        print(f"  • Success Rate: {wide_row['Success_Rate']:.1f}%")
        
        print(f"\n10,000 Synthetic Assets Dataset:")
        print(f"  • Average Return: {synthetic_row['Avg_Return']:.4f}")
        print(f"  • Execution Time: {synthetic_row['Avg_Time']:.2f}s") 
        print(f"  • Success Rate: {synthetic_row['Success_Rate']:.1f}%")
        
        # Performance ratios
        time_ratio = synthetic_row['Avg_Time'] / wide_row['Avg_Time']
        return_ratio = synthetic_row['Avg_Return'] / wide_row['Avg_Return'] if wide_row['Avg_Return'] != 0 else float('inf')
        
        print(f"\nScaling Analysis:")
        print(f"  • Asset count increased by: {10000/680:.1f}x")
        print(f"  • Execution time increased by: {time_ratio:.1f}x") 
        print(f"  • Return performance ratio: {return_ratio:.2f}x")
        
else:
    print("No comparison data available")

#### Dataset Comparison Summary

In [ ]:
# Performance analysis for D-Wave CQM QUBO optimization (Synthetic Dataset - 10,000 columns)
dwave_cqm_synthetic_solvers = [
    {'path_template': '../../results/qubo/results_synthetic_{}_dwave_cqm.pkl', 'name': 'D-Wave CQM QUBO (Synthetic)'},
]

print("\n" + "="*100)
print("COMPREHENSIVE PERFORMANCE SUMMARY - D-Wave CQM QUBO Optimization (Synthetic Dataset - 10,000 columns)")
print("="*100)
summary_df = performance_summary(dwave_cqm_synthetic_solvers, n_periods)
if not summary_df.empty:
    display(summary_df)
    
    # Performance analysis
    print("\nPerformance Analysis:")
    print("-" * 50)
    print(f"Successful periods: {summary_df.iloc[0]['Periods']}/{n_periods}")
    print(f"Average return: {summary_df.iloc[0]['Avg_Return']:.4f}")
    print(f"Average execution time: {summary_df.iloc[0]['Avg_Time']:.2f}s")
    print(f"Success rate: {summary_df.iloc[0]['Success_Rate']:.1f}%")
else:
    print("No summary data available")

#### Performance Analysis for Synthetic Dataset

In [ ]:
results_path_dwave_cqm_synthetic = '../../results/qubo/results_synthetic_{}_dwave_cqm.pkl'
_ = run_experiment(results_path_dwave_cqm_synthetic, source_synthetic, benchmark, opt_fun_dwave_cqm, parameters)

#### D-Wave CQM QUBO Optimization - Synthetic Dataset

In [ ]:
# Load the synthetic dataset
source_synthetic = pd.read_pickle('../../data/synthetic_close.pkl')
print(f"Synthetic dataset shape: {source_synthetic.shape}")
print(f"Value range: {source_synthetic.min().min():.4f} to {source_synthetic.max().max():.4f}")
print(f"Sample columns: {source_synthetic.columns[:10].tolist()}")
source_synthetic.head()

## Test with Synthetic Dataset (10,000 columns)

Run the same analysis with the expanded synthetic dataset